# Matching `(material, species)` tuples to the `sead_staging` taxa tables

`explore.ipynb` extracted every unique `(material, species)` pair from the C14 DataFrame into `output/strucke_v8_material_species_unique.csv`. This notebook checks whether the Swedish-language `species` values in that table can be tied back to the taxonomic hierarchy stored in the `sead_staging` Postgres database, and — since many of those values are generic terms ("björk", "tall") rather than full binomial names — at *which* taxonomic rank (species / genus / family / order) that connection is actually possible.

In [ ]:
import os
import re
from collections import Counter

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

load_dotenv()

DB_HOST = os.environ["DB_HOST"]
DB_PORT = os.environ["DB_PORT"]
DB_NAME = os.environ["DB_NAME"]
DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [4]:
material_species = pd.read_csv('output/strucke_v8_material_species_unique.csv')
print(material_species.shape)
material_species.head(10)

(755, 3)


,material,species,n_rows
0,NaN,NaN,1
1,Animalier,Fiskfjäll,1
2,Animalier,Hjorthår,1
3,Animalier,"Hjärnsubstans, människa",1
4,Animalier,Hår,1
5,Animalier,Läder,6
6,Animalier,Nötfiber,1
7,Animalier,Nöthår,4
8,Animalier,Renhår,1
9,Animalier,Tagel,2


## What taxa-related tables exist in `sead_staging`?

In [5]:
taxa_tables = pd.read_sql("""
    select table_schema, table_name
    from information_schema.tables
    where table_name ilike '%%taxa%%' or table_name ilike '%%taxon%%'
    order by table_schema, table_name
""", engine)
taxa_tables

,table_schema,table_name
0,facet,abundance_taxon_shortcut
1,facet,geochronology_taxa_shortcut
2,facet,view_abundances_by_taxon_analysis_entity
3,facet,view_taxa_biblio
4,postgrest_default_api,imported_taxa_replacement
5,postgrest_default_api,site_sample_taxon_abundance
6,postgrest_default_api,taxa_common_name
7,postgrest_default_api,taxa_image
8,postgrest_default_api,taxa_measured_attribute
9,postgrest_default_api,taxa_reference_specimen


The `public` schema (same classic SEAD `tbl_` naming used by `tbl_biblio` in `explore.ipynb`) holds a full taxonomic hierarchy:

- `tbl_taxa_tree_orders` → `tbl_taxa_tree_families` → `tbl_taxa_tree_genera` → `tbl_taxa_tree_master` (species)
- `tbl_taxa_common_names` links a species-level `taxon_id` to a vernacular name tagged with a `language_id`
- `tbl_languages` — checking it confirms `language_id = 2` is Swedish, matching the language used in the DataFrame's `species` column
- `view_taxa_alphabetically` is a convenience view with the whole order → family → genus → species chain pre-joined, used throughout below

In [6]:
pd.read_sql('select * from public.tbl_languages', engine)

,language_id,date_updated,language_name_english,language_name_native
0,1,2012-09-21 16:51:47.967181+00:00,English,English
1,2,2012-09-21 16:51:47.967181+00:00,Swedish,Svenska


In [7]:
pd.read_sql('select * from public.view_taxa_alphabetically order by genus, species limit 8', engine)

,order_id,order,family_id,family,genus_id,genus,taxon_id,species,author_id,author
0,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,40931,ater,4594.0,(Villers)
1,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,42028,ater,5181.0,(Vill.)
2,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,29294,parallelepipedus,3236.0,(Pill. & Mitt.)
3,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,44831,parallelopipedus,4776.0,auctt misspelling
4,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,29295,parallelus,3113.0,(Duft.)
5,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,29296,sp.,NaN,NaN
6,138,ORDER PENDING CLASSIFICATION,1647,CARABIDAE,12711,Abax,44830,striola,3104.0,(F.)
7,138,ORDER PENDING CLASSIFICATION,1732,SERROPALPIDAE,13672,Abdera,45030,?bifasciata,3168.0,(Marsham)


## Scope check: flora only, or flora + fauna?

The DataFrame's `material` column includes bone (`Ben, brända`, `Ben, obrända`), tooth (`Tand`), horn, and fish material, so some `species` values name animals rather than plants (e.g. `Får`, `Get`, `Häst`, `Älg`, `Nöt`). Before trying to match anything, it's worth checking whether the taxa tables cover fauna at all.

In [8]:
orders = pd.read_sql('select order_id, order_name from public.tbl_taxa_tree_orders order by order_name', engine)
print(f'{len(orders)} taxonomic orders in the database:')
print(orders['order_name'].tolist())

animal_check = pd.read_sql("""
    select cn.common_name, v.genus, v.family, v."order"
    from public.tbl_taxa_common_names cn
    join public.view_taxa_alphabetically v on v.taxon_id = cn.taxon_id
    where lower(cn.common_name) in ('får','get','häst','hund','svin','älg','nöt','nötkreatur','ren','katt','gris','ko','fisk')
""", engine)
print(f'\n{len(animal_check)} matches for common domestic-animal names')
animal_check

57 taxonomic orders in the database:
['Alismatales', 'Apiales', 'Aquifoliales', 'Arecales', 'Asparagales', 'Asterales', 'Brassicales', 'Capparales', 'Caryophyllales', 'Ceratophyllales', 'Cerealia', 'Commelinales', 'Cornales', 'Cucurbitales', 'Dipsacales', 'Equisetales', 'Ericales', 'Fabales', 'Fagales', 'Gentianales', 'Geraniales', 'Insertis Sedis', 'Isoetales', 'Lamiales', 'Laurales', 'Liliales', 'Lycopodiales', 'Lythrales', 'Magnoliales', 'Malpighiales', 'Malvales', 'Marsileales', 'Myrtales', 'No data', 'Nymphales', 'Ophioglossales', 'ORDER PENDING CLASSIFICATION', 'Osmundales', 'Oxalidales', 'Pinales', 'Piperales', 'Planta', 'Poales', 'Polypodiales', 'Primates', 'Proteales', 'Ranunculales', 'Rosales', 'Salicales', 'Salvinales', 'Santalales', 'Sapindales', 'Saxifragales', 'Selaginellales', 'Solanales', 'Sphagnales', 'Zingiberales']

0 matches for common domestic-animal names


,common_name,genus,family,order


Every one of the 57 orders is a plant order (plus a handful of placeholders: `Cerealia` for cereals, `Insertis Sedis`, `No data`, `ORDER PENDING CLASSIFICATION`, `Planta`). None of the common domestic-animal names return a hit.

**The taxa tables in `sead_staging` are flora-only.** Bone/tooth/horn rows whose `species` value names an animal (får, get, häst, älg, nöt, …) have nothing to link to here and are out of scope for the matching below — that's a gap in the *database*, not something the DataFrame values can be blamed for.

One more caveat worth flagging while poking around the hierarchy: in this snapshot, genus `Acer` (lönn / maple) is linked to family `Acoraceae` / order `Alismatales`, which is biologically wrong (Acer belongs in Sapindaceae / Sapindales). Spot-checking a dozen other common genera below shows correct placements, so this looks like an isolated stale foreign key rather than a systemic problem — flagged here for awareness, not fixed (fixing the source DB is out of scope for this notebook).

In [9]:
sanity_check = pd.read_sql("""
    select g.genus_name, f.family_name, o.order_name
    from public.tbl_taxa_tree_genera g
    join public.tbl_taxa_tree_families f on f.family_id = g.family_id
    join public.tbl_taxa_tree_orders o on o.order_id = f.order_id
    where g.genus_name in ('Acer','Betula','Alnus','Corylus','Pinus','Picea','Quercus','Fagus','Salix','Populus','Tilia')
    order by g.genus_name
""", engine)
sanity_check

,genus_name,family_name,order_name
0,Acer,Acoraceae,Alismatales
1,Alnus,Betulaceae,Fagales
2,Betula,Betulaceae,Fagales
3,Corylus,Corylaceae,Fagales
4,Fagus,Fagaceae,Fagales
5,Picea,Pinaceae,Pinales
6,Pinus,Pinaceae,Pinales
7,Populus,Salicaceae,Salicales
8,Quercus,Fagaceae,Fagales
9,Salix,Salicaceae,Salicales


## Matching strategy

The DataFrame's `species` values are messy in a few specific ways:

- **compound cells**: `"Al, Björk"`, `"Asp/Salix sp"`, `"Ask, hassel, Pomoideae, Populus sp"` — several taxa crammed into one field, joined by `,` / `/` / `och` / `&`
- **mixed vocabulary**: mostly Swedish common names (`björk`, `tall`, `ek`), but some already Latin (`Salix sp`, `Populus sp`, `cf Pinus sp`, `Maloideae`)
- **non-taxonomic values**: plant parts (`bark`, `näver`, `barr`, `kottefjäll`), and text that isn't a taxon at all (`Amulettring`, `Bivax`)

So, mirroring the author-name normalization done earlier in `explore.ipynb`: split every unique `species` value into individual tokens once, match each token against the DB hierarchy, then re-attach the results to the original `(material, species)` rows. Token frequency is weighted by `n_rows` so the coverage stats reflect actual row counts, not just unique-combo counts.

Matching is layered from most to least specific:

1. **exact Swedish common name** → resolves to a species (`tbl_taxa_common_names`, `language_id=2`)
2. **exact Latin name**, after stripping `cf.` / `sp.` / `indet.` qualifiers → genus / family / order
3. **inferred genus from common-name suffix** — Swedish tree names are compounds with the general term last (`gråal`, `klibbal` → `*al` → *Alnus*; `vårtbjörk`, `glasbjörk` → `*björk` → *Betula*). If every Swedish common name ending in the token belongs to the same genus, that's accepted as a genus-level match; if it resolves to more than one genus (e.g. `*alm` also catches `kokospalm`/`dadelpalm`), it's flagged as ambiguous with candidates listed rather than guessed.

Anything left over doesn't match the taxonomy tables at all, and is flagged for manual review.

In [10]:
sv_common = pd.read_sql("""
    select lower(cn.common_name) as common_name_lc, cn.common_name,
           v.taxon_id, v.species, v.genus, v.family, v."order"
    from public.tbl_taxa_common_names cn
    join public.view_taxa_alphabetically v on v.taxon_id = cn.taxon_id
    where cn.language_id = 2
""", engine)
genera = pd.read_sql('select lower(genus_name) as genus_lc, genus_name, genus_id, family_id from public.tbl_taxa_tree_genera', engine)
families = pd.read_sql('select lower(family_name) as family_lc, family_name, family_id, order_id from public.tbl_taxa_tree_families', engine)
orders_lookup = pd.read_sql('select lower(order_name) as order_lc, order_name, order_id from public.tbl_taxa_tree_orders', engine)

print(f'{len(sv_common)} Swedish common names ({sv_common["common_name_lc"].duplicated().sum()} case-insensitive duplicates)')

common_map = sv_common.drop_duplicates('common_name_lc').set_index('common_name_lc')
genus_hierarchy = (
    genera.merge(families[['family_id', 'family_name', 'order_id']], on='family_id', how='left')
          .merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
          .drop_duplicates('genus_lc').set_index('genus_lc')
)
family_hierarchy = (
    families.merge(orders_lookup[['order_id', 'order_name']], on='order_id', how='left')
             .drop_duplicates('family_lc').set_index('family_lc')
)
order_hierarchy = orders_lookup.drop_duplicates('order_lc').set_index('order_lc')

4272 Swedish common names (2 case-insensitive duplicates)


In [11]:
SPLIT_RE = re.compile(r'\s*(?:,|/|\bresp\.?\b|\bsamt\b|\boch\b|&)\s*', re.I)

def tokenize(species_value):
    return [p.strip(' ?.') for p in SPLIT_RE.split(species_value) if p.strip(' ?.')]

token_counts = Counter()
for species_value, n in material_species.dropna(subset=['species'])[['species', 'n_rows']].itertuples(index=False):
    for tok in tokenize(species_value):
        token_counts[tok] += n

print(f'{material_species["species"].dropna().nunique()} unique species values -> {len(token_counts)} unique tokens')

583 unique species values -> 460 unique tokens


In [12]:
LATIN_QUALIFIER_RE = re.compile(r'^(cf\.?\s+|aff\.?\s+)|(\s+(sp\.?|spp\.?|indet\.?)\s*$)', re.I)

def strip_latin_qualifiers(token):
    prev = None
    while prev != token:
        prev = token
        token = LATIN_QUALIFIER_RE.sub('', token).strip()
    return token

def match_exact(token_lc):
    if token_lc in common_map.index:
        rec = common_map.loc[token_lc]
        return dict(match_level='species (common name)', genus=rec['genus'], family=rec['family'], order=rec['order'],
                    detail=f"{rec['genus']} {rec['species']}")
    cleaned = strip_latin_qualifiers(token_lc)
    if cleaned in genus_hierarchy.index:
        rec = genus_hierarchy.loc[cleaned]
        return dict(match_level='genus (latin)', genus=rec['genus_name'], family=rec['family_name'], order=rec['order_name'],
                    detail=rec['genus_name'])
    if cleaned in family_hierarchy.index:
        rec = family_hierarchy.loc[cleaned]
        return dict(match_level='family (latin)', genus=None, family=rec['family_name'], order=rec['order_name'],
                    detail=rec['family_name'])
    if cleaned in order_hierarchy.index:
        rec = order_hierarchy.loc[cleaned]
        return dict(match_level='order (latin)', genus=None, family=None, order=rec['order_name'], detail=rec['order_name'])
    return None

def infer_genus_by_suffix(token_lc):
    """Swedish tree names compound as <modifier><base>, e.g. 'klibbal' -> Alnus. Only
    accepted when every common name ending in the token converges on a single genus."""
    if len(token_lc) < 2 or not token_lc.isalpha():
        return None
    hits = sv_common[sv_common['common_name_lc'].str.endswith(token_lc)]
    if hits.empty:
        return None
    candidate_genera = hits['genus'].unique().tolist()
    if len(candidate_genera) == 1:
        rec = genus_hierarchy[genus_hierarchy['genus_name'] == candidate_genera[0]]
        return dict(match_level='genus (suffix-inferred)', genus=candidate_genera[0],
                    family=rec['family_name'].iloc[0] if len(rec) else None,
                    order=rec['order_name'].iloc[0] if len(rec) else None,
                    detail=f'{len(hits)} common names -> {candidate_genera[0]}', needs_review=False)
    return dict(match_level='genus (suffix, ambiguous)', genus=None, family=None, order=None,
                detail=f"candidates: {', '.join(candidate_genera[:6])}", needs_review=True)

In [13]:
rows = []
for token, n in token_counts.items():
    token_lc = token.lower()
    result = match_exact(token_lc) or infer_genus_by_suffix(token_lc) or dict(match_level=None, genus=None, family=None, order=None, detail=None)
    result.setdefault('needs_review', result['match_level'] is None)
    result['token'] = token
    result['n_rows'] = n
    rows.append(result)

token_matches = pd.DataFrame(rows)[['token', 'n_rows', 'match_level', 'genus', 'family', 'order', 'detail', 'needs_review']]
token_matches = token_matches.sort_values('n_rows', ascending=False).reset_index(drop=True)

summary = (
    token_matches.groupby('match_level', dropna=False)['n_rows']
    .agg(['count', 'sum']).rename(columns={'count': 'n_tokens', 'sum': 'n_rows'})
)
summary['pct_row_weighted_tokens'] = (summary['n_rows'] / token_matches['n_rows'].sum() * 100).round(1)
summary.sort_values('n_rows', ascending=False)

,n_tokens,n_rows,pct_row_weighted_tokens
match_level,,,
species (common name),77,9418,50.2
NaN,291,4291,22.9
genus (suffix-inferred),25,2165,11.5
"genus (suffix, ambiguous)",29,1884,10.0
genus (latin),38,1016,5.4


Highest-frequency unresolved tokens — worth a manual look (plant parts, animal names, and non-taxonomic text are expected here):

In [14]:
token_matches[token_matches['match_level'].isna()].head(30)

,token,n_rows,match_level,genus,family,order,detail,needs_review
5,Människa,1170,NaN,NaN,NaN,NaN,NaN,True
7,Skalkorn,870,NaN,NaN,NaN,NaN,NaN,True
13,Matskorpa,281,NaN,NaN,NaN,NaN,NaN,True
14,Lövträd,228,NaN,NaN,NaN,NaN,NaN,True
18,Nötkreatur,120,NaN,NaN,NaN,NaN,NaN,True
22,Däggdjur,88,NaN,NaN,NaN,NaN,NaN,True
23,Maloideae,86,NaN,NaN,NaN,NaN,NaN,True
24,Häst,85,NaN,NaN,NaN,NaN,NaN,True
26,Barrträd,69,NaN,NaN,NaN,NaN,NaN,True
27,Emmer,66,NaN,NaN,NaN,NaN,NaN,True


Ambiguous suffix-inferred candidates — ends in the token but resolves to more than one genus, needs a human to pick:

In [15]:
token_matches[token_matches['match_level'] == 'genus (suffix, ambiguous)']

,token,n_rows,match_level,genus,family,order,detail,needs_review
4,Al,1251,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Lathyrus, Alnus",True
9,Korn,433,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Impatiens, Hordelymus, Hordeum",True
28,Alm,59,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Cocos, Phoenix, Ulmus",True
44,Enbär,32,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Rubus, Physalis",True
47,Nöt,29,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Acicarpha, Arachis, Juglans, Ptero...",True
55,Ört,19,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Laserpitium, Achillea, Artemisia, ...",True
70,Ull,11,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Nymphoides, Polemonium, Eriophorum",True
79,Starr,9,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Carex, Kobresia",True
89,Gräs,7,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Scheuchzeria, Isoetes, Pilularia, ...",True
97,Ärta,6,"genus (suffix, ambiguous)",NaN,NaN,NaN,"candidates: Dicentra, Lamprocapnos",True


## Rolling the token matches back up to `(material, species)` rows

For each original `species` value, look up every token's match and keep the most specific level reached, plus which genera/families/orders it resolved to.

In [16]:
token_lookup = token_matches.set_index('token')[['match_level', 'genus', 'family', 'order']]
LEVEL_RANK = {
    'species (common name)': 4, 'genus (latin)': 3, 'genus (suffix-inferred)': 3,
    'family (latin)': 2, 'order (latin)': 1, 'genus (suffix, ambiguous)': 0,
}

def summarize_species_value(species_value):
    toks = tokenize(species_value)
    if not toks:
        return pd.Series({'n_tokens': 0, 'n_matched': 0, 'best_match_level': None,
                           'matched_genera': None, 'matched_families': None, 'matched_orders': None})
    levels, genera_hit, fam_hit, ord_hit, matched = [], set(), set(), set(), 0
    for t in toks:
        info = token_lookup.loc[t] if t in token_lookup.index else None
        if info is not None and pd.notna(info['match_level']) and LEVEL_RANK.get(info['match_level'], -1) > 0:
            matched += 1
            levels.append(info['match_level'])
            if pd.notna(info['genus']):
                genera_hit.add(info['genus'])
            if pd.notna(info['family']):
                fam_hit.add(info['family'])
            if pd.notna(info['order']):
                ord_hit.add(info['order'])
    best = max(levels, key=lambda l: LEVEL_RANK.get(l, -1)) if levels else None
    return pd.Series({
        'n_tokens': len(toks), 'n_matched': matched, 'best_match_level': best,
        'matched_genera': ', '.join(sorted(genera_hit)) or None,
        'matched_families': ', '.join(sorted(fam_hit)) or None,
        'matched_orders': ', '.join(sorted(ord_hit)) or None,
    })

match_summary = material_species['species'].fillna('').apply(summarize_species_value)
material_species_matched = pd.concat([material_species, match_summary], axis=1)
material_species_matched.sort_values('n_rows', ascending=False).head(20)

,material,species,n_rows,n_tokens,n_matched,best_match_level,matched_genera,matched_families,matched_orders
437,Träkol,NaN,9959,0.0,0.0,NaN,NaN,NaN,NaN
605,Träkol,Tall,2589,1.0,1.0,species (common name),Pinus,Pinaceae,Pinales
480,Träkol,Björk,1897,1.0,1.0,genus (suffix-inferred),Betula,Betulaceae,Fagales
507,Träkol,Ek,1652,1.0,1.0,species (common name),Quercus,Fagaceae,Fagales
440,Träkol,Al,1172,1.0,0.0,NaN,NaN,NaN,NaN
542,Träkol,Hassel,1108,1.0,1.0,species (common name),Corylus,Corylaceae,Fagales
532,Träkol,Gran,1039,1.0,1.0,species (common name),Picea,Pinaceae,Pinales
250,Förkolnat frö,Skalkorn,859,1.0,0.0,NaN,NaN,NaN,NaN
60,"Ben, brända",Människa,621,1.0,0.0,NaN,NaN,NaN,NaN
366,Skalfragment,Hassel,525,1.0,1.0,species (common name),Corylus,Corylaceae,Fagales


In [17]:
tagged = material_species_matched[material_species_matched['species'].notna()]
total_rows = tagged['n_rows'].sum()
covered_rows = tagged.loc[tagged['n_matched'] > 0, 'n_rows'].sum()
fully_covered_rows = tagged.loc[tagged['n_matched'] == tagged['n_tokens'], 'n_rows'].sum()

print(f'rows with a species value tagged:    {total_rows:,}')
print(f'rows with >=1 taxon token resolved:  {covered_rows:,} ({covered_rows/total_rows:.1%})')
print(f'rows with ALL taxon tokens resolved: {fully_covered_rows:,} ({fully_covered_rows/total_rows:.1%})')
print()
material_species_matched['best_match_level'].value_counts(dropna=False)

rows with a species value tagged:    18,240
rows with >=1 taxon token resolved:  12,444 (68.2%)
rows with ALL taxon tokens resolved: 12,217 (67.0%)



best_match_level
NaN                        432
species (common name)      224
genus (latin)               50
genus (suffix-inferred)     49
Name: count, dtype: int64

In [ ]:
token_matches.to_csv('output/species_taxa_token_matches.csv', index=False, encoding='utf-8-sig')
material_species_matched.to_csv('output/material_species_taxa_matches.csv', index=False, encoding='utf-8-sig')

## Conclusions

- **~68% of tagged rows** (row-weighted) can be connected to the `sead_staging` taxa tables through at least one token in their `species` value; **~67%** have *every* token in the value resolved.
- Direct **species-level** matches via `tbl_taxa_common_names` (language_id=2) alone cover about half the row-weighted tokens — these are the clean, single-word entries (`tall` → *Pinus sylvestris*, `ek` → *Quercus robur*, `gran` → *Picea abies*, …).
- A meaningful share only resolves at **genus level**, confirming the premise of this notebook: several of the most common values (`björk`, `al`, `alm`, …) are genus-level Swedish generics with *no* single matching species-level common name in the DB (multiple wild species share the vernacular name), so `tbl_taxa_tree_genera` — not `tbl_taxa_tree_master` — is the right join target for them. The suffix-inference step recovers these automatically when the mapping is unambiguous, and flags it for manual review otherwise (e.g. `*alm` collides with `kokospalm`/`dadelpalm`).
- **No family- or order-only matches were needed in practice** — every Latin value in the data that wasn't already resolvable at genus level (e.g. `Cerealia`, via the special cereals catch-all genus) turned out to have a genus-level home. The family/order matching code path is still worth keeping for messier future data.
- The taxa tables are **flora-only** — animal `species` values (`får`, `get`, `häst`, `älg`, `nöt`, …), which come from bone/tooth/horn `material` rows, have no match target in `sead_staging` at all and account for a chunk of the unresolved rows.
- The rest of the unresolved rows are genuinely non-taxonomic `species` values — plant parts (`bark`, `barr`, `kottefjäll`, `näver`), unspecific descriptors (`lövträd`, `örtfragment`), or unrelated text (`Amulettring`, `Bivax`) — that were never going to match a taxonomy table and should be filtered out (or handled separately) rather than "fixed" here.

Two review files were saved for follow-up, mirroring the pattern of `output/author_normalization_review.csv`:

- `output/species_taxa_token_matches.csv` — one row per unique token, with its matched genus/family/order and a `needs_review` flag
- `output/material_species_taxa_matches.csv` — the original `(material, species)` tuples with the best taxonomic rank reached and the resolved genus/family/order names attached

## v2: attaching the exact SEAD common name and species binomial

Follow-up request: extend the row-level match table with the *exact* Swedish common name and Latin species binomial that SEAD matched on. This is only meaningful where the match reached full **species** level (`species (common name)`) — anything resolved just at genus/family/order level leaves these two columns `null`, since no single species can be attributed there without guessing.

In [ ]:
def sead_species_lookup(token):
    token_lc = token.lower()
    if token_lc in common_map.index:
        rec = common_map.loc[token_lc]
        return rec['common_name'], f"{rec['genus']} {rec['species']}"
    return None, None

token_matches[['sead_common_name', 'sead_species_name']] = token_matches['token'].apply(
    lambda t: pd.Series(sead_species_lookup(t))
)
print(f"tokens with a species-level SEAD match: {token_matches['sead_species_name'].notna().sum()}")
token_matches[token_matches['sead_species_name'].notna()].head(10)

In [ ]:
token_lookup_v2 = token_matches.set_index('token')[
    ['match_level', 'genus', 'family', 'order', 'sead_common_name', 'sead_species_name']
]

def summarize_species_value_v2(species_value):
    toks = tokenize(species_value)
    if not toks:
        return pd.Series({'sead_common_name': None, 'sead_species_name': None})
    common_names_hit, species_names_hit = [], []
    for t in toks:
        info = token_lookup_v2.loc[t] if t in token_lookup_v2.index else None
        if info is None:
            continue
        if pd.notna(info['sead_common_name']) and info['sead_common_name'] not in common_names_hit:
            common_names_hit.append(info['sead_common_name'])
        if pd.notna(info['sead_species_name']) and info['sead_species_name'] not in species_names_hit:
            species_names_hit.append(info['sead_species_name'])
    return pd.Series({
        'sead_common_name': '; '.join(common_names_hit) or None,
        'sead_species_name': '; '.join(species_names_hit) or None,
    })

sead_fields = material_species['species'].fillna('').apply(summarize_species_value_v2)
material_species_matched_v2 = pd.concat([material_species_matched, sead_fields], axis=1)

print(f"rows with sead_species_name populated: {material_species_matched_v2['sead_species_name'].notna().sum()} / {len(material_species_matched_v2)}")
material_species_matched_v2[material_species_matched_v2['sead_species_name'].notna()].sort_values('n_rows', ascending=False).head(15)

In [ ]:
material_species_matched_v2.to_csv('output/material_species_taxa_match_v2.csv', index=False, encoding='utf-8-sig')

One thing this surfaces that wasn't visible in v1's genus-only columns: the suffix-inference heuristic occasionally produces a plausible-looking but wrong genus for short tokens that collide with an unrelated Swedish plant name — e.g. `Animalier` / `Hår` (animal hair) resolves to genus *Drosera* (sundew; Swedish `sileshår` ends in `hår`), and `Animalier` / `Läder` (leather) resolves to *Sambucus* (elder; `fläder`). These correctly get **no** `sead_species_name` (they're only genus-level matches), but their `matched_genera` value is spurious — `Animalier` rows are never going to hit anything real in a flora-only taxonomy, so that whole material category is safer to exclude outright rather than trust the genus-level guess.

`output/material_species_taxa_match_v2.csv` now carries every column from v1 plus:

- `sead_common_name` — the exact Swedish common name(s) SEAD matched on (only when a token resolved at species level; `;`-joined if a compound cell had more than one)
- `sead_species_name` — the corresponding Latin binomial(s) from `tbl_taxa_tree_master` / `tbl_taxa_tree_genera` (same null/join rule)